In [ ]:
# Cell 1 - bootstrap (plan 2.1). The ONLY place environment setup lives.
# Replace YOUR_GIT_REMOTE_URL with this repo's clone URL.
import os

from google.colab import drive

REPO = "YOUR_GIT_REMOTE_URL"
DRIVE = "/content/drive/MyDrive/minissp"

drive.mount('/content/drive')
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"
!git clone $REPO /content/minissp 2>/dev/null || (cd /content/minissp && git pull)
%cd /content/minissp
# First session only - build the Linux wheel cache into Drive:
# !pip download -r requirements.txt -d $DRIVE/wheels
!pip install -q --no-index --find-links=$DRIVE/wheels -r requirements.txt
!pip install -q -e . --no-deps
!python --version && pip freeze > $DRIVE/requirements.lock


In [ ]:
# Cell 2 - Phase 2 proof: policy + judge + retriever resident, 8+8 trajectories.
# Read runs/smoke/samples/smoke.jsonl before running cell 3. A format_valid_rate
# near 0 is a prompt/chat-template bug; training will not fix it.
!python -m minissp.rollout --smoke --run-id smoke --n 8 \
    --index-dir $DRIVE/index --out runs/smoke

In [ ]:
# Cell 3 - the run (plan 4.2). No --resume flag: runs/t4-001/latest decides.
# Re-running this cell after a disconnect continues from the last checkpoint.
!python -m minissp.train --run-id t4-001 --max-steps 50 \
    --index-dir $DRIVE/index --runs-root $DRIVE/runs